<a href="https://colab.research.google.com/github/JA-RgueZ/FEDRO128/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
from fastapi import FastAPI, HTTPException
from google.oauth2 import service_account
import gspread

app = FastAPI(title="FEDRO API v1.0")

# --- CONFIGURACIÓN DE IDENTIDAD DE DATOS ---
def get_sheets_client():
    creds_json = os.environ.get("GOOGLE_CREDS_JSON")
    if not creds_json:
        raise ValueError("GOOGLE_CREDS_JSON no configurada")

    info_json = json.loads(creds_json)

    print("credencial google ok")

    scopes = [
        'https://www.googleapis.com/auth/spreadsheets',
        'https://www.googleapis.com/auth/drive'
    ]

    creds = service_account.Credentials.from_service_account_info(
        info_json,
        scopes=scopes  # Aplicar los scopes aquí
    )
    return gspread.authorize(creds)

# --- ENDPOINT DE BÚSQUEDA (ETAPA 2) ---
@app.get("/auth/perfil/{telefono}")
def get_perfil(telefono: str):
    try:
        client = get_sheets_client()

        # 1. Abre el archivo por su nombre
        spreadsheet = client.open("FEDRO128")

        # 2. Selecciona específicamente la pestaña "Cuadro"
        sheet = spreadsheet.worksheet("Cuadro")

        # 3. Busca el teléfono (asegúrate que el formato en la sheet sea texto o coincida)
        cell = sheet.find(telefono)

        print("tengo el sheet")

        if not cell:
            return {"identificado": False, "error": "QH no encontrado"}

        # 4. Extraer datos del QH (Asumiendo: A=RUT, B=Nombre, C=Celular, D=Grado)
        row = sheet.row_values(cell.row)

        return {
            "identificado": True,
            "rut": row[0],
            "nombre_completo": row[1],
            "grado": int(row[3]),
            "status": "Regular"
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error: {str(e)}")
